# NumPy: The Foundation of Scientific Computing in Python

**Version:** Comprehensive Self-Study Guide  
**Level:** Beginner → Advanced  
**Prerequisites:** Basic Python knowledge

---

## What is NumPy?

NumPy (Numerical Python) is the fundamental package for scientific computing in Python. It provides:

- A powerful **N-dimensional array** object (`ndarray`)
- Sophisticated **broadcasting** capabilities
- Comprehensive **mathematical functions** (linear algebra, Fourier transforms, random number generation)
- **C-level performance** via vectorized operations

## Why NumPy?

| Feature | Python List | NumPy Array |
| ------- | ----------- | ----------- |
| Memory | ~16 bytes/element | ~8 bytes/element (contiguous) |
| Speed | Slow (loops in Python) | Fast (C loops, vectorized) |
| Element-wise ops | Manual loops | `arr * 2` (one call) |
| Data type | Homogeneous (objects) | Homogeneous (typed) |
| Dimensions | 1D only (nested lists) | N-dimensional |
| Broadcasting | Not supported | Full support |

## Memory Layout ASCII

```
Python List:          NumPy Array (C-contiguous):
┌────┬────┬────┬────┐  ┌────┬────┬────┬────┐
│ptr0│ptr1│ptr2│ptr3│  │ 1  │ 2  │ 3  │ 4  │
└─┬──┴─┬──┴─┬──┴─┬──┘  └────┴────┴────┴────┘
  │    │    │    │      [int64, contiguous in RAM]
  ▼    ▼    ▼    ▼
 [1]  [2]  [3]  [4]     (scattered in heap)
```

In [ ]:
# Standard import convention
import numpy as np

# Check version
print(f"NumPy version: {np.__version__}")

---
## Section 1: Array Creation

NumPy provides multiple ways to create arrays. Choosing the right method improves both readability and performance.

### 1.1 `np.array()`

**Definition:**  
Creates an ndarray from any array-like object (list, tuple, or another array).

**Intuition:**  
Think of `np.array()` as a *conversion factory* — it takes Python data structures and casts them into high-performance NumPy arrays.

**Syntax:**
```python
np.array(object, dtype=None, *, copy=True, order='K', subok=False, ndmin=0)
```

**Parameters:**

| Parameter | Type | Default | Description |
| --------- | ---- | ------- | ----------- |
| `object` | array_like | (required) | Input data: list, tuple, nested sequences, or another array |
| `dtype` | dtype, optional | `None` | Desired output data type. If `None`, inferred from data |
| `copy` | bool | `True` | If `True` (default), the object is copied. If `False`, a view is returned when possible |
| `order` | {'K', 'A', 'C', 'F'} | `'K'` | Memory layout: 'C' (row-major), 'F' (column-major), 'A' (any), 'K' (keep) |
| `subok` | bool | `False` | If `True`, subclasses of ndarray are passed through |
| `ndmin` | int | `0` | Minimum number of dimensions; array is broadcast to meet this |

**Returns:**  
`numpy.ndarray` — a new array containing the converted data.

**Time Complexity:**  
O(n) — where n is the number of elements.

**Common Mistakes:**
- Forgetting that mixed-type lists produce `dtype=object` arrays (slow)
- Assuming `copy=False` guarantees no copy — it only avoids a copy when the input is already an array with matching dtype
- Confusing `ndmin` with `.reshape()` — `ndmin` pads dimensions on the *left*

**Best Practices:**
- Always specify `dtype` for numeric data to avoid type inference surprises
- Use `copy=False` only when you are certain the input won't be mutated
- For large data, prefer `np.fromiter()` or `np.frombuffer()` for memory efficiency

**Real-World Applications:**
- Loading CSV rows into arrays for ML pipelines
- Converting image pixel lists into ndarrays
- Initializing neural network weight matrices from flat data

In [ ]:
# --- EASY EXAMPLES (5) ---

# Easy 1: 1D array from a list
a1 = np.array([1, 2, 3, 4, 5])
print("Easy 1:", a1)

# Easy 2: 2D array from nested lists
a2 = np.array([[1, 2], [3, 4]])
print("Easy 2:\n", a2)

# Easy 3: Specify dtype explicitly
a3 = np.array([1, 2, 3], dtype='float32')
print("Easy 3:", a3, "dtype:", a3.dtype)

# Easy 4: From a tuple
a4 = np.array((10, 20, 30))
print("Easy 4:", a4)

# Easy 5: ndmin parameter
a5 = np.array([1, 2, 3], ndmin=2)
print("Easy 5:", a5, "shape:", a5.shape)

In [ ]:
# --- INTERMEDIATE EXAMPLES (3) ---

# Intermediate 1: Nested irregular sequences → 1D object array
irregular = np.array([[1, 2], [3, 4, 5]], dtype=object)
print("Inter 1:", irregular, "dtype:", irregular.dtype, "shape:", irregular.shape)

# Intermediate 2: copy=False vs copy=True
original = np.array([1, 2, 3])
view_try = np.array(original, copy=False)
view_try[0] = 99
print("Inter 2: original changed?", original[0])  # 99 because copy=False means view

# Intermediate 3: subok=True
class MyArray(np.ndarray): pass
sub = np.array(MyArray([1, 2, 3]), subok=True)
print("Inter 3: subtype?", type(sub))

In [ ]:
# --- ADVANCED EXAMPLES (2) ---

# Advanced 1: Building an array from a generator (memory efficient)
def gen():
    for i in range(10):
        yield i ** 2
arr_from_gen = np.fromiter(gen(), dtype='int64')
print("Adv 1:", arr_from_gen)

# Advanced 2: Memory-mapped array creation (for huge datasets)
# np.memmap('data.dat', dtype='float32', mode='write', shape=(1000, 1000))
print("Adv 2: Use np.memmap for >RAM datasets (commented out)")

### 1.2 `np.zeros()` & `np.ones()`

**Definition:**  
Create arrays filled with 0 or 1, respectively, of a given shape.

**Intuition:**  
These are the most common *initialization* functions — used to allocate memory with a constant fill value.

**Syntax:**
```python
np.zeros(shape, dtype=float, order='C')
np.ones(shape, dtype=float, order='C')
```

**Parameters:**

| Parameter | Type | Default | Description |
| --------- | ---- | ------- | ----------- |
| `shape` | int or tuple of ints | (required) | Shape of the new array (e.g., `3` → 1D, `(2,3)` → 2D) |
| `dtype` | dtype | `float64` | Desired output data type |
| `order` | {'C', 'F'} | `'C'` | Memory layout (row-major vs column-major) |

**Returns:**  
`numpy.ndarray` — filled with 0s or 1s.

**Time Complexity:**  
O(n) — the array is allocated and filled.

**Common Mistakes:**
- Specifying `shape` as a list instead of tuple — both work, but tuples are idiomatic
- Forgetting the default dtype is `float64`, not `int64`
- Using `np.zeros(3,2)` instead of `np.zeros((3,2))` — the shape must be a *single* argument

**Best Practices:**
- Specify `dtype=int` if you need integer arrays to avoid unnecessary float conversion
- Pre-allocate with `np.zeros` + assignment instead of `np.append` in loops

**Real-World Applications:**
- Initializing bias vectors in neural networks (`np.zeros`)
- Creating masks for image processing (`np.ones`)
- Allocating feature matrices in ML pipelines

In [ ]:
# --- EASY EXAMPLES (5) ---

z1 = np.zeros(5)
print("Easy 1 (zeros 1D):", z1)

z2 = np.zeros((2, 3))
print("Easy 2 (zeros 2D):\n", z2)

z3 = np.ones(4)
print("Easy 3 (ones 1D):", z3)

z4 = np.ones((2, 2), dtype=int)
print("Easy 4 (ones int):\n", z4)

z5 = np.ones((3, 3, 3))  # 3D array of ones
print("Easy 5 (ones 3D shape):", z5.shape)

In [ ]:
# --- INTERMEDIATE EXAMPLES (3) ---

# Inter 1: Order matters for large arrays (performance)
c_order = np.zeros((1000, 1000), order='C')
f_order = np.zeros((1000, 1000), order='F')
print("Inter 1: C order strides:", c_order.strides, "F order strides:", f_order.strides)

# Inter 2: Using zeros_like / ones_like to match shape
template = np.array([[1, 2], [3, 4]])
match = np.zeros_like(template, dtype=float)
print("Inter 2: zeros_like:\n", match)

# Inter 3: ones with complex dtype
complex_ones = np.ones(3, dtype=complex)
print("Inter 3: complex ones:", complex_ones)

In [ ]:
# --- ADVANCED EXAMPLES (2) ---

# Adv 1: Pre-allocate then fill (much faster than np.append in loop)
N = 100000
buf = np.zeros(N, dtype=float)
for i in range(N):
    buf[i] = i ** 0.5
print("Adv 1: First 5 buf elements:", buf[:5])

# Adv 2: Using np.zeros with structured dtype
struct_dtype = np.dtype([('x', 'f4'), ('y', 'f4'), ('label', 'U10')])
structured_arr = np.zeros(5, dtype=struct_dtype)
print("Adv 2: Structured arr:\n", structured_arr)

### 1.3 `np.empty()`

**Definition:**  
Returns a new array without initializing entries (contains whatever garbage is in memory).

**Intuition:**  
Like renting an apartment — you get the space immediately but the previous tenant's stuff is still there. Faster than `np.zeros` for very large arrays.

**Syntax:**
```python
np.empty(shape, dtype=float, order='C')
```

**Parameters:**

| Parameter | Type | Default | Description |
| --------- | ---- | ------- | ----------- |
| `shape` | int or tuple | (required) | Shape of the array |
| `dtype` | dtype | `float64` | Desired output dtype |
| `order` | {'C', 'F'} | `'C'` | Memory layout |

**Returns:**  
`numpy.ndarray` — uninitialized data.

**Time Complexity:**  
O(1) for allocation (no fill). Compare: `np.zeros` is O(n).

**Common Mistakes:**
- Reading from `np.empty` before writing — values are unpredictable (security risk for sensitive data)
- Assuming it's initialized to 0

**Best Practices:**
- Use only when you immediately overwrite every element
- Use `np.zeros` by default unless profiling shows `empty` is a bottleneck

**Real-World Applications:**
- Buffer arrays in real-time signal processing
- Output arrays for C-extension functions that fill data

In [ ]:
# Examples
e1 = np.empty(5)
print("Empty 1D (values are garbage):", e1)

e2 = np.empty((2, 2), dtype=int)
print("Empty 2D int:\n", e2)

# After immediately writing, it's safe
e3 = np.empty(1000)
e3[:] = np.arange(1000)
print("Empty then fill (first 5):", e3[:5])

### 1.4 `np.arange()`

**Definition:**  
Returns evenly spaced values within a given interval (like Python's `range()` but returns an ndarray).

**Intuition:**  
The *sequence generator* — creates arithmetic progressions in one call.

**Syntax:**
```python
np.arange([start,] stop[, step], dtype=None)
```

**Parameters:**

| Parameter | Type | Default | Description |
| --------- | ---- | ------- | ----------- |
| `start` | number | `0` | Start of interval (inclusive) |
| `stop` | number | (required with 1 arg) | End of interval (exclusive) |
| `step` | number | `1` | Spacing between values |
| `dtype` | dtype | `None` (inferred) | Output dtype |

**Returns:**  
`numpy.ndarray` — evenly spaced values.

**Time Complexity:**  
O(n) where n = ceil((stop - start) / step).

**Common Mistakes:**
- `stop` is *exclusive*, not inclusive
- Using integer `step` with float bounds — dtype inference can surprise
- `arange(0, 1, 0.1)` accumulates floating-point error; use `np.linspace` for exact control

**Best Practices:**
- Use `np.linspace` instead of `np.arange` when you need a specific number of points
- For integer sequences, rely on inferred dtype (int64) instead of specifying dtype

**Real-World Applications:**
- Generating index arrays for loops
- Creating time-step vectors for plotting
- Producing epoch sequences in training loops

In [ ]:
# --- EASY EXAMPLES (5) ---

r1 = np.arange(5)
print("Easy 1 (stop only):", r1)

r2 = np.arange(2, 10)
print("Easy 2 (start, stop):", r2)

r3 = np.arange(0, 20, 3)
print("Easy 3 (with step):", r3)

r4 = np.arange(1.0, 5.0, 0.5)
print("Easy 4 (float step):", r4)

r5 = np.arange(10, dtype='float32')
print("Easy 5 (explicit dtype):", r5.dtype)

In [ ]:
# --- INTERMEDIATE EXAMPLES (3) ---

# Inter 1: Negative step (descending)
r6 = np.arange(10, 0, -1)
print("Inter 1 (descending):", r6)

# Inter 2: arange with dtype=np.complex64
r7 = np.arange(1, 5, dtype=complex)
print("Inter 2 (complex):", r7)

# Inter 3: Nested arange for mesh-like indices
x = np.arange(3)
y = np.arange(3)[:, np.newaxis]
print("Inter 3 (grid coords):\n", x + y * 10)

In [ ]:
# --- ADVANCED EXAMPLES (2) ---

# Adv 1: Floating point precision issue
bad = np.arange(0, 1, 0.1)
print("Adv 1 (FP error):", bad)  # may show 0.30000000000000004
good = np.linspace(0, 1, 11)
print("     Fixed with linspace:", good)

# Adv 2: 2D grid from arange (DIY meshgrid)
xs = np.arange(0, 3)
ys = np.arange(0, 2)[:, None]
grid = np.dstack(np.broadcast_arrays(xs, ys))[0]
print("Adv 2 (grid):\n", grid)

### 1.5 `np.linspace()`

**Definition:**  
Returns `num` evenly spaced samples over the interval `[start, stop]` (inclusive of both ends by default).

**Intuition:**  
Unlike `arange` (which uses a step size), `linspace` uses a *count* — you specify how many points you want.

**Syntax:**
```python
np.linspace(start, stop, num=50, endpoint=True, retstep=False, dtype=None, axis=0)
```

**Parameters:**

| Parameter | Type | Default | Description |
| --------- | ---- | ------- | ----------- |
| `start` | array_like | (required) | Starting value |
| `stop` | array_like | (required) | Ending value |
| `num` | int | `50` | Number of samples to generate |
| `endpoint` | bool | `True` | If `True`, `stop` is included; if `False`, excluded |
| `retstep` | bool | `False` | If `True`, return `(samples, step_size)` |
| `dtype` | dtype | `None` | Output dtype (inferred from inputs) |
| `axis` | int | `0` | Axis along which to lay out the result (for array start/stop) |

**Returns:**  
`numpy.ndarray` — shape `(num,)` for scalar start/stop.  
If `retstep=True`: `(ndarray, float)`.

**Time Complexity:**  
O(num).

**Common Mistakes:**
- Forgetting that `num` defaults to 50 (not the interval length)
- Using `endpoint=False` and expecting the same count — the last point before `stop` is included

**Best Practices:**
- Use `linspace` for plotting axes (guarantees even spacing)
- Use `retstep=True` when you need the step size for numerical integration

**Real-World Applications:**
- Generating time vectors for signal processing (`np.linspace(0, T, fs)`)
- Creating smooth curves for plotting functions
- Defining grid points for numerical PDE solvers

In [ ]:
# --- EASY EXAMPLES (5) ---

l1 = np.linspace(0, 10, 5)
print("Easy 1:", l1)

l2 = np.linspace(0, 1, 5, endpoint=False)
print("Easy 2 (no endpoint):", l2)

l3 = np.linspace(0, 10, 5, retstep=True)
print("Easy 3 (with step):", l3)

l4 = np.linspace(0, 100, 11, dtype=int)
print("Easy 4 (int):", l4)

l5 = np.linspace(-np.pi, np.pi, 7)
print("Easy 5 (symmetric):", l5)

In [ ]:
# --- INTERMEDIATE EXAMPLES (3) ---

# Inter 1: Array start/stop (vectorized linspace)
starts = np.array([0, 10, 100])
stops = np.array([5, 20, 200])
multi = np.linspace(starts, stops, num=4, axis=1)
print("Inter 1 (multi-linspace):\n", multi)

# Inter 2: 2D grid with meshgrid + linspace
x = np.linspace(-2, 2, 5)
y = np.linspace(-2, 2, 5)
xx, yy = np.meshgrid(x, y)
zz = np.sin(xx**2 + yy**2)
print("Inter 2 (meshgrid shape):", zz.shape)

# Inter 3: Non-linear spacing via transformation
log_spaced = np.exp(np.linspace(np.log(1), np.log(1000), 10))
print("Inter 3 (log-spaced):", np.round(log_spaced, 2))

In [ ]:
# --- ADVANCED EXAMPLES (2) ---

# Adv 1: 3D grid using linspace + mgrid (advanced)
grid_3d = np.mgrid[-1:1:5j, -1:1:5j, -1:1:5j]  # 5j = 5 steps via linspace
print("Adv 1 (3D grid shape):", grid_3d.shape)

# Adv 2: linspace for numerical integration (trapezoidal)
t = np.linspace(0, np.pi, 1000)
dt = t[1] - t[0]
integral = np.trapz(np.sin(t), t)
print(f"Adv 2: ∫sin(x)dx from 0 to π ≈ {integral:.6f} (expected: 2.0)")

### 1.6 `np.eye()` — Identity Matrix

**Definition:**  
Returns a 2D array with 1s on the diagonal and 0s elsewhere.

**Intuition:**  
The identity matrix Iₙ — the multiplicative identity in linear algebra.

**Syntax:**
```python
np.eye(N, M=None, k=0, dtype=float, order='C')
```

**Parameters:**

| Parameter | Type | Default | Description |
| --------- | ---- | ------- | ----------- |
| `N` | int | (required) | Number of rows |
| `M` | int, optional | `None` (=N) | Number of columns |
| `k` | int | `0` | Diagonal offset: 0=main, positive=upper, negative=lower |
| `dtype` | dtype | `float` | Output dtype |

**Returns:**  
`numpy.ndarray` — shape `(N, M)`.

**Time Complexity:**  
O(N × M).

**Common Mistakes:**
- Mistaking `np.eye(3)` for `np.identity(3)` — identical, but `eye` is more flexible (offset, rectangular)
- Forgetting `k` parameter exists — useful for banded matrices

**Real-World Applications:**
- Initializing weight matrices in neural networks
- Ridge regression (adding λI to XᵀX)
- Creating permutation matrices with `k` offset

In [ ]:
print("eye(3):\n", np.eye(3))
print("\neye(3, 5):\n", np.eye(3, 5))
print("\neye(3, k=1):\n", np.eye(3, k=1))
print("\neye(3, k=-1):\n", np.eye(3, k=-1))

### 1.7 `np.full()`

**Definition:**  
Returns a new array of given shape, filled with a constant value.

**Syntax:**
```python
np.full(shape, fill_value, dtype=None, order='C')
```

**Parameters:**
| Parameter | Type | Default | Description |
| --------- | ---- | ------- | ----------- |
| `shape` | int or tuple | (required) | Shape of output |
| `fill_value` | scalar or array_like | (required) | Fill value |
| `dtype` | dtype | `None` | Output dtype (inferred if None) |

**Returns:**  
`numpy.ndarray` — filled with `fill_value`.

**Time Complexity:**  
O(n).

**Example:**

In [ ]:
f1 = np.full((3, 3), 7)
print("All 7s:\n", f1)

f2 = np.full(5, np.pi, dtype='float32')
print("Pi 5 times:", f2)

f3 = np.full((2, 4), 'Hello')
print("String fill:\n", f3)

### 1.8 `np.random` Module — Random Array Creation

**Intuition:**  
NumPy's random module is the backbone of stochastic operations in ML — weight initialization, data shuffling, synthetic data generation.

#### `np.random.rand()`

**Definition:**  
Uniform distribution over `[0, 1)`.

**Syntax:**
```python
np.random.rand(d0, d1, ..., dn)
```
**Parameters:**  
Dimensions as positional args (not a tuple!).  
**Returns:**  
`numpy.ndarray` — shape `(d0, d1, ..., dn)`.

**Example:**

In [ ]:
print("rand(3):", np.random.rand(3))
print("rand(2,3):\n", np.random.rand(2, 3))

#### `np.random.randn()`

**Definition:**  
Standard normal distribution (mean=0, variance=1).

**Syntax:**
```python
np.random.randn(d0, d1, ..., dn)
```

**Example:**

In [ ]:
print("randn(1000) mean:", np.random.randn(1000).mean())  # ≈ 0
print("randn(1000) std:", np.random.randn(1000).std())    # ≈ 1

#### `np.random.randint()`

**Definition:**  
Random integers from `low` (inclusive) to `high` (exclusive).

**Syntax:**
```python
np.random.randint(low, high=None, size=None, dtype=int)
```

**Parameters:**
| Parameter | Type | Default | Description |
| --------- | ---- | ------- | ----------- |
| `low` | int | (required) | Lowest integer (or highest if `high` is None) |
| `high` | int | `None` | One above the largest integer |
| `size` | int or tuple | `None` | Output shape |
| `dtype` | dtype | `int` | Output dtype |

**Example:**

In [ ]:
print("randint(10):", np.random.randint(10))  # single int
print("randint(0, 10, 5):", np.random.randint(0, 10, 5))
print("randint(0, 10, (2,3)):\n", np.random.randint(0, 10, (2, 3)))

---
## Section 2: Array Attributes

Once an array is created, these attributes tell you everything about its structure.

### Attribute Reference Table

| Attribute | Description | Return Type | Example (`arr = np.array([[1,2],[3,4]])`) |
| --------- | ----------- | ----------- | ----------------------------------------- |
| `arr.shape` | Tuple of dimension lengths | `tuple of int` | `(2, 2)` |
| `arr.ndim` | Number of dimensions | `int` | `2` |
| `arr.size` | Total number of elements | `int` | `4` |
| `arr.dtype` | Data type of elements | `dtype` object | `int64` |
| `arr.itemsize` | Bytes per element | `int` | `8` |
| `arr.nbytes` | Total bytes consumed | `int` | `32` |
| `arr.T` | Transpose (2D only) | `ndarray` | `[[1,3],[2,4]]` |
| `arr.strides` | Bytes to step per axis | `tuple of int` | `(16, 8)` |
| `arr.data` | Memory buffer | `memoryview` | `<memory...>` |
| `arr.flat` | 1D iterator over array | `numpy.flatiter` | `[1, 2, 3, 4]` |
| `arr.real` / `arr.imag` | Real/imaginary parts | `ndarray` | — |

In [ ]:
# Create a sample array to inspect attributes
arr = np.array([[1, 2, 3], [4, 5, 6]], dtype='float64')

print("Array:\n", arr)
print("shape:", arr.shape)
print("ndim:", arr.ndim)
print("size:", arr.size)
print("dtype:", arr.dtype)
print("itemsize:", arr.itemsize, "bytes")
print("nbytes:", arr.nbytes, "bytes")
print("T:\n", arr.T)
print("strides:", arr.strides)

# Using flat
print("Flat:", [x for x in arr.flat])

### Understanding Strides — ASCII Memory Layout

Strides define how NumPy walks through memory:

```
C-contiguous (row-major):
  arr = [[1, 2],
         [3, 4]]
  
  Memory: | 1 | 2 | 3 | 4 |
  Index:    0   1   2   3
  
  strides[0] = 2 * 8 = 16 (rows: skip 2 elements)
  strides[1] = 1 * 8 = 8  (cols: skip 1 element)

F-contiguous (column-major):
  Memory: | 1 | 3 | 2 | 4 |
  strides[0] = 1 * 8 = 8
  strides[1] = 2 * 8 = 16
```

---
## Section 3: Indexing & Slicing

NumPy offers rich indexing beyond Python lists. Mastering these is essential for efficient data manipulation.

### 3.1 Basic Indexing

**Syntax:** `arr[i, j, k]` — same as Python lists but with commas for each dimension.

**Intuition:**  
Unlike nested lists `arr[i][j]`, NumPy uses `arr[i, j]` — one indexing operation, faster and cleaner.

In [ ]:
arr = np.array([[10, 20, 30], [40, 50, 60], [70, 80, 90]])
print("Array:\n", arr)
print("arr[1, 2]:", arr[1, 2])   # 60
print("arr[0]:", arr[0])         # first row
print("arr[-1, -1]:", arr[-1, -1])  # 90 (last row, last col)

### 3.2 Slicing

**Syntax:** `arr[start:stop:step, start:stop:step, ...]`

**Key Rule:** Slices return **views** (no copy), so modifying a slice modifies the original!

In [ ]:
# 2D slicing
arr = np.arange(25).reshape(5, 5)
print("Original:\n", arr)
print("\nFirst 2 rows, first 3 cols:\n", arr[:2, :3])
print("\nEvery other row, every other col:\n", arr[::2, ::2])
print("\nRow 0 (all cols):", arr[0, :])
print("\nCol 0 (all rows):", arr[:, 0])

# SLICING RETURNS A VIEW
view = arr[:2, :2]
view[0, 0] = 999
print("\nAfter modifying view, original changed?", arr[0, 0] == 999)

### 3.3 Fancy Indexing (Integer Array Indexing)

**Definition:**  
Passing arrays of indices to select multiple elements at once.

**Important:** Fancy indexing returns a **copy**, not a view.

In [ ]:
arr = np.arange(10)
indices = np.array([1, 3, 5, 7])
print("Fancy index:", arr[indices])  # [1 3 5 7]

# 2D fancy indexing
arr2d = np.arange(16).reshape(4, 4)
rows = np.array([0, 2, 3])
cols = np.array([1, 2, 0])
print("\n2D fancy:\n", arr2d[rows, cols])  # [1, 10, 12]

# Fancy indexing returns a COPY
arr = np.array([1, 2, 3, 4])
copy = arr[[0, 2]]
copy[0] = 999
print("\nOriginal (unchanged):", arr)

### 3.4 Boolean Indexing

**Definition:**  
Using a boolean array of the same shape to filter elements.

**Powerful for:** Conditional selection, masking, outlier removal.

In [ ]:
arr = np.array([1, 5, 2, 8, 3, 9, 4])
mask = arr > 4
print("Mask:", mask)
print("Filtered (>4):", arr[mask])

# In-place modification with mask
arr[arr % 2 == 0] = -1
print("\nEven replaced by -1:", arr)

# 2D boolean indexing
arr2d = np.arange(12).reshape(3, 4)
mask2d = arr2d % 3 == 0
print("\n2D mask:\n", mask2d)
print("Elements divisible by 3:", arr2d[mask2d])

### 3.5 `np.where()` — Conditional Selection

**Syntax:**
```python
np.where(condition, [x, y])  # or np.where(condition) → indices
```

**Two Modes:**
1. `np.where(cond, x, y)` — return elements from `x` where cond is True, `y` where False
2. `np.where(cond)` — return tuple of indices where cond is True

**Example:**

In [ ]:
arr = np.array([1, -2, 3, -4, 5])

# Mode 1: clamp negative to 0
clamped = np.where(arr < 0, 0, arr)
print("Clamped:", clamped)

# Mode 2: get indices of positives
pos_indices = np.where(arr > 0)
print("Positive indices:", pos_indices)

# 2D where
arr2d = np.array([[1, -1], [-2, 3]])
row_idx, col_idx = np.where(arr2d > 0)
print("2D positive at:", list(zip(row_idx, col_idx)))

---
## Section 4: Array Manipulation

Functions to change shape, combine, split, and reorder arrays.

### 4.1 `reshape()` & `resize()`

**`arr.reshape(new_shape)`:**  
Returns a **view** (if possible) with the new shape. Total elements must match.

**`np.resize(arr, new_shape)`:**  
Returns a **copy** with the new shape. Elements are repeated/truncated to fill.

**Syntax:**
```python
arr.reshape(shape, order='C')
np.resize(arr, new_shape)
```

**ASCII:**
```
reshape (1D → 2D):
  [1 2 3 4 5 6]  ──→  [[1 2 3]
                          [4 5 6]]
```

In [ ]:
arr = np.arange(12)

# reshape
r = arr.reshape(3, 4)
print("Reshape (3,4):\n", r)

# -1 is inferred
r2 = arr.reshape(2, -1)
print("\nReshape (2,-1):\n", r2)

# resize (repeats data if needed)
small = np.array([1, 2, 3])
big = np.resize(small, (3, 3))
print("\nResize (repeated):\n", big)

# order matters
print("\nReshape F-order:\n", arr.reshape(3, 4, order='F'))

### 4.2 `flatten()` vs `ravel()`

| Method | Returns | Memory | Modifying affects original? |
| ------ | ------- | ------ | --------------------------- |
| `arr.flatten()` | Copy | New memory | No |
| `arr.ravel()` | View (usually) | Shared | Yes (if view) |
| `arr.flat` | Iterator | — | Yes (via setitem) |

**Best Practice:** Use `ravel()` when you want speed and don't mind views. Use `flatten()` when you need a guaranteed independent copy.

In [ ]:
arr = np.array([[1, 2], [3, 4]])

flat_copy = arr.flatten()
flat_view = arr.ravel()

flat_copy[0] = 999
print("After modifying copy, original:\n", arr)  # unchanged

flat_view[0] = 999
print("After modifying view, original:\n", arr)  # changed!

### 4.3 Transpose & Axis Operations

**`arr.T`** — transpose of 2D array (shortcut).  
**`arr.transpose(*axes)`** — permute axes of any-dimensional array.  
**`np.swapaxes(arr, axis1, axis2)`** — swap two axes.  
**`np.moveaxis(arr, source, destination)`** — move axes to new positions.

In [ ]:
# 2D transpose
arr = np.array([[1, 2, 3], [4, 5, 6]])
print("T:\n", arr.T)  # shape (3,2)

# 3D transpose
arr3d = np.arange(24).reshape(2, 3, 4)
print("\nOriginal shape:", arr3d.shape)
print("Transpose (1,2,0):", arr3d.transpose(1, 2, 0).shape)

# swapaxes
print("\nSwap axes:", np.swapaxes(arr3d, 0, 1).shape)

### 4.4 Concatenation: `concatenate`, `vstack`, `hstack`, `stack`

| Function | Axis | Description | Example Shapes |
| -------- | ---- | ----------- | -------------- |
| `np.concatenate` | tuple | Join along existing axis | `(3,4)+(3,4) → (6,4)` |
| `np.vstack` | 0 | Stack vertically (rows) | `(3,)+(3,) → (6,)` |
| `np.hstack` | 1 | Stack horizontally (cols) | `(3,4)+(3,5) → (3,9)` |
| `np.dstack` | 2 | Stack depth-wise | `(3,4)+(3,4) → (3,4,2)` |
| `np.stack` | new axis | Stack along NEW axis | `3×(3,4) → (3,3,4)` |

**Common Mistake:** `concatenate` needs a **tuple** of arrays, and all dimensions except the join axis must match.

In [ ]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])

print("vstack:\n", np.vstack((a, b)))
print("\nhstack:\n", np.hstack((a, b)))
print("\nconcatenate axis=0:\n", np.concatenate((a, b), axis=0))
print("\nconcatenate axis=1:\n", np.concatenate((a, b), axis=1))
print("\nstack (new axis):\n", np.stack((a, b), axis=0).shape)  # (2, 2, 2)

### 4.5 Splitting: `split`, `vsplit`, `hsplit`

**Syntax:**
```python
np.split(arr, indices_or_sections, axis=0)
np.vsplit(arr, sections)   # equivalent to split(axis=0)
np.hsplit(arr, sections)   # equivalent to split(axis=1)
```

**Example:**

In [ ]:
arr = np.arange(16).reshape(4, 4)
print("Original:\n", arr)

# Split into 2 equal parts along rows
parts = np.split(arr, 2, axis=0)
print("\nSplit into 2 (rows):")
for p in parts:
    print(p)

# hsplit
left, right = np.hsplit(arr, 2)
print("\nLeft half:\n", left)
print("\nRight half:\n", right)

# Split at specific indices
parts = np.split(arr, [1, 3], axis=0)
print("\nSplit at [1,3]:")
for i, p in enumerate(parts):
    print(f"Part {i}:\n{p}\n")

---
## Section 5: Vectorized Operations & Broadcasting

**Key Principle:** "Never write loops over NumPy arrays." Vectorized ops are 10-100x faster than Python loops.

### 5.1 Arithmetic Operations

All arithmetic operators work **element-wise**:

In [ ]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

print("a + b:", a + b)       # addition
print("a * b:", a * b)       # element-wise multiply
print("a ** 2:", a ** 2)     # power
print("a @ b:", a @ b)       # dot product (1D)
print("a / b:", a / b)       # division
print("a // b:", a // b)     # floor division
print("a % b:", a % b)       # modulus

# Speed comparison
import time
big = np.arange(10**7)
t0 = time.perf_counter()
result = big * 2  # vectorized
t_vec = time.perf_counter() - t0

t0 = time.perf_counter()
result = [x * 2 for x in big]  # list comprehension
t_py = time.perf_counter() - t0
print(f"\nVectorized: {t_vec:.4f}s, Python loop: {t_py:.4f}s ({t_py/t_vec:.0f}x faster)")

### 5.2 Broadcasting

**Definition:**  
NumPy's ability to perform operations on arrays of **different shapes** by stretching the smaller one without copying data.

**The Broadcasting Rule:**  
Two dimensions are compatible when they are **equal**, or **one of them is 1**.

**ASCII Diagram:**
```
Array A: (3, 1)    Array B: (3,)  →  Result: (3, 3)
┌───┐              ┌───┬───┬───┐    ┌───┬───┬───┐
│ 1 │  +  [4,5,6]  →  │ 4 │ 5 │ 6 │  →  │ 5 │ 6 │ 7 │
├───┤              └───┴───┴───┘    ├───┼───┼───┤
│ 2 │                              │ 6 │ 7 │ 8 │
├───┤                              ├───┼───┼───┤
│ 3 │                              │ 7 │ 8 │ 9 │
└───┘                              └───┴───┴───┘
```

**Steps:**
1. Align shapes from the right
2. For each dimension: size must match or one must be 1
3. Missing dimensions are treated as 1

In [ ]:
# Broadcasting examples
a = np.array([[1], [2], [3]])   # shape (3, 1)
b = np.array([4, 5, 6])         # shape (3,) → broadcast to (3, 3)
print("Broadcast result:\n", a + b)

# Add a scalar to all elements
print("\nScalar broadcast:", np.array([1, 2, 3]) + 100)

# Center data (subtract mean along axis, keepdims)
data = np.random.randn(3, 4)
centered = data - data.mean(axis=0, keepdims=True)
print("\nCentered (mean near 0):", centered.mean(axis=0).round(10))

# Broadcasting rules: (3,1) + (1,4) → (3,4)
row = np.array([1, 2, 3, 4]).reshape(1, 4)
col = np.array([10, 20, 30]).reshape(3, 1)
print("\nRow + Col broadcast:\n", row + col)

### 5.3 Comparison Operators & Boolean Arrays

Comparisons return boolean arrays, which are useful for masking:

In [ ]:
arr = np.array([1, 5, 2, 8, 3, 9, 4, 7])

print("arr > 4:", arr > 4)
print("any > 8?", np.any(arr > 8))
print("all > 0?", np.all(arr > 0))
print("Count > 4:", np.sum(arr > 4))

# Chaining (use &, |, ~ instead of and, or, not)
print("Between 3 and 7:", arr[(arr > 3) & (arr < 7)])
print("Outside 2 to 8:", arr[~(arr >= 2) & (arr <= 8)])

---
## Section 6: Universal Functions (ufuncs)

**Definition:**  
ufuncs operate **element-wise** on ndarrays with C-level speed. There are ~60+ ufuncs in NumPy.

**Intuition:**  
Every ufunc is a "vectorized wrapper" around a C function that operates on single values.

### 6.1 Trigonometric Functions

| Function | Description | Domain |
| -------- | ----------- | ------ |
| `np.sin` | Sine | (-∞, ∞) → [-1, 1] |
| `np.cos` | Cosine | (-∞, ∞) → [-1, 1] |
| `np.tan` | Tangent | All but π/2 + kπ |
| `np.arcsin` | Arc sine | [-1, 1] → [-π/2, π/2] |
| `np.arctan2(y, x)` | 4-quadrant arctan | — |

**Common Mistake:** Trigonometric functions expect **radians**, not degrees. Use `np.deg2rad()` to convert.

In [ ]:
angles = np.array([0, 30, 45, 60, 90])  # degrees
rad = np.deg2rad(angles)

print("sin:", np.round(np.sin(rad), 4))
print("cos:", np.round(np.cos(rad), 4))
print("tan:", np.round(np.tan(rad), 4))

# arctan2
print("\narctan2(0, 1):", np.arctan2(0, 1))      # 0
print("arctan2(1, 0):", np.arctan2(1, 0))      # π/2
print("arctan2(-1, -1):", np.arctan2(-1, -1))  # -3π/4

### 6.2 Exponential & Logarithmic

| Function | Description |
| -------- | ----------- |
| `np.exp(x)` | eˣ |
| `np.expm1(x)` | eˣ - 1 (accurate for small x) |
| `np.log(x)` | Natural log ln(x) |
| `np.log2(x)` | Base-2 log |
| `np.log10(x)` | Base-10 log |
| `np.log1p(x)` | ln(1 + x) (accurate for small x) |

In [ ]:
x = np.array([1, 2, np.e, 10, 100])

print("exp:", np.exp(x))
print("log:", np.log(x))
print("log10:", np.log10(x))
print("log2:", np.log2(x))

# Numerically stable softmax
def softmax(x):
    e_x = np.exp(x - np.max(x))  # subtract max for numerical stability
    return e_x / e_x.sum()

scores = np.array([1000, 1005, 990])
print("\nStable softmax:", softmax(scores))

### 6.3 Rounding Functions

| Function | Description | Example |
| -------- | ----------- | ------- |
| `np.round(x, decimals=0)` | Round to given decimals | `np.round(3.14159, 2)` → `3.14` |
| `np.floor(x)` | Round down | `np.floor(3.9)` → `3.0` |
| `np.ceil(x)` | Round up | `np.ceil(3.1)` → `4.0` |
| `np.trunc(x)` | Truncate toward 0 | `np.trunc(-3.9)` → `-3.0` |
| `np.rint(x)` | Round to nearest int | `np.rint(2.5)` → `2.0` (banker's rounding) |

In [ ]:
vals = np.array([1.49, 1.50, 1.51, -1.49, -1.50])
print("Original:", vals)
print("round:", np.round(vals))
print("floor:", np.floor(vals))
print("ceil:", np.ceil(vals))
print("trunc:", np.trunc(vals))

---
## Section 7: Statistical Operations

Aggregation functions operate on the entire array or along a specified axis.

### Summary Statistics Table

| Function | Description | Output (no axis) | NaN-safe version |
| -------- | ----------- | ---------------- | ---------------- |
| `arr.sum()` / `np.sum(arr)` | Sum of all elements | scalar | `np.nansum` |
| `arr.mean()` | Arithmetic mean | scalar | `np.nanmean` |
| `arr.std()` | Standard deviation | scalar | `np.nanstd` |
| `arr.var()` | Variance | scalar | `np.nanvar` |
| `arr.min()` / `arr.max()` | Min / max | scalar | `np.nanmin`/`np.nanmax` |
| `arr.argmin()` / `arr.argmax()` | Index of min/max | int | `np.nanargmin`/`np.nanargmax` |
| `arr.cumsum()` | Cumulative sum | same shape | `np.nancumsum` |
| `arr.cumprod()` | Cumulative product | same shape | `np.nancumprod` |
| `np.percentile(arr, q)` | Qth percentile | scalar or array | `np.nanpercentile` |
| `np.median(arr)` | Median | scalar | `np.nanmedian` |

In [ ]:
# 2D array
data = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("Data:\n", data)

print("Sum all:", data.sum())
print("Sum axis=0 (cols):", data.sum(axis=0))
print("Sum axis=1 (rows):", data.sum(axis=1))
print("Mean:", data.mean())
print("Std:", data.std())
print("Min (global):", data.min())
print("Argmax (global):", data.argmax())
print("Cumsum flattened:", data.cumsum())

# Percentile
print("25th percentile:", np.percentile(data, 25))
print("Median:", np.median(data))

In [ ]:
# --- NaN-safe operations
noisy = np.array([1.0, 2.0, np.nan, 4.0, np.nan])
print("mean (nan unsafe):", np.mean(noisy))       # nan
print("nanmean:", np.nanmean(noisy))              # 2.333...
print("nansum:", np.nansum(noisy))                # 7.0

# Keep dimensions
arr = np.arange(12).reshape(3, 4)
row_means = arr.mean(axis=1, keepdims=True)
print("\nRow means (keepdims):\n", row_means)

---
## Section 8: Linear Algebra

NumPy's `linalg` submodule provides all essential linear algebra operations.

### 8.1 Matrix Multiplication

| Method | Description | Example |
| ------ | ----------- | ------- |
| `np.dot(a, b)` | Dot product (1D) or matrix multiply (2D) | `np.dot(A, B)` |
| `a @ b` | `@` operator (Python 3.5+) — recommended | `A @ B` |
| `np.matmul(a, b)` | Matrix multiplication (broadcasting on batch dims) | `np.matmul(A, B)` |
| `np.inner(a, b)` | Inner product (last axes) | — |
| `np.outer(a, b)` | Outer product | — |

**Rule:** `(m, n) @ (n, p) → (m, p)`

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print("A @ B:\n", A @ B)
print("\nnp.dot(A, B):\n", np.dot(A, B))

# Vector dot product
v = np.array([1, 2, 3])
w = np.array([4, 5, 6])
print("\nDot product:", v @ w)  # 32

# Outer product
print("\nOuter product:\n", np.outer(v, w))

### 8.2 `np.linalg` — Key Functions

| Function | Description | Formula |
| -------- | ----------- | ------- |
| `np.linalg.inv(A)` | Matrix inverse | A⁻¹ |
| `np.linalg.det(A)` | Determinant | det(A) |
| `np.linalg.eig(A)` | Eigenvalues & eigenvectors | Av = λv |
| `np.linalg.svd(A)` | Singular value decomposition | UΣV* |
| `np.linalg.solve(A, b)` | Solve linear system Ax = b | x = A⁻¹b |
| `np.linalg.qr(A)` | QR decomposition | A = QR |
| `np.linalg.cholesky(A)` | Cholesky decomposition | A = LL* (A must be SPD) |
| `np.linalg.norm(x)` | Matrix/vector norm | ‖x‖ |
| `np.linalg.matrix_rank(A)` | Rank of matrix | rank(A) |
| `np.linalg.cond(A)` | Condition number | κ(A) |

In [ ]:
# Inverse & Determinant
A = np.array([[4, 7], [2, 6]])
print("A:\n", A)
print("det(A):", np.linalg.det(A))
print("inv(A):\n", np.linalg.inv(A))
print("A @ inv(A):\n", A @ np.linalg.inv(A))  # ≈ identity

# Solve linear system: 3x + y = 9, x + 2y = 8
A = np.array([[3, 1], [1, 2]])
b = np.array([9, 8])
x = np.linalg.solve(A, b)
print("\nSolution x:", x)  # [2, 3]
print("Check:", A @ x)     # [9, 8]

In [ ]:
# Eigenvalues & eigenvectors
A = np.array([[3, 1], [1, 3]])
eigvals, eigvecs = np.linalg.eig(A)
print("Eigenvalues:", eigvals)
print("Eigenvectors (columns):\n", eigvecs)
print("Check A*v = λ*v:")
for i in range(2):
    lhs = A @ eigvecs[:, i]
    rhs = eigvals[i] * eigvecs[:, i]
    print(f"  v{i}: match?", np.allclose(lhs, rhs))

# SVD
A = np.array([[1, 2], [3, 4], [5, 6]])
U, S, Vt = np.linalg.svd(A)
print("\nSVD shapes:", U.shape, S.shape, Vt.shape)
print("Singular values:", S)
print("Reconstructed (U @ diag(S) @ Vt):\n", U @ np.diag(S) @ Vt)

### 8.3 Matrix Norms

**Syntax:** `np.linalg.norm(x, ord=None, axis=None)`

| `ord` | Vector norm | Matrix norm |
| ----- | ----------- | ----------- |
| `None` | L2 (Euclidean) | Frobenius |
| `1` | L1 (sum of abs) | Max column sum |
| `2` | L2 (Euclidean) | Spectral norm (largest SV) |
| `np.inf` | Max absolute value | Max row sum |
| `-np.inf` | Min absolute value | Min row sum |
| `'fro'` | — | Frobenius |

In [ ]:
v = np.array([3, -4])
print("L0 (count non-zero):", np.count_nonzero(v))
print("L1 norm:", np.linalg.norm(v, ord=1))           # 7
print("L2 norm:", np.linalg.norm(v))                  # 5
print("Infinity norm:", np.linalg.norm(v, ord=np.inf))  # 4

# Matrix norms
M = np.array([[1, 2], [3, -4]])
print("\nFrobenius norm:", np.linalg.norm(M))
print("Spectral norm:", np.linalg.norm(M, ord=2))

---
## Section 9: Random Number Generation

**Best Practice (NumPy ≥ 1.17):** Use the `default_rng()` interface instead of the legacy `np.random.*`.

### 9.1 Legacy `np.random.*` vs New Generator API

| Feature | Legacy (`np.random.*`) | New (`np.random.default_rng()`) |
| ------- | ---------------------- | ------------------------------- |
| Global state | Yes (shared) | No (per-generator) |
| Thread safety | Poor | Good |
| Speed | Baseline | ~2x faster |
| Bit generators | MT19937 only | PCG64, Philox, SFC64, etc. |
| Recommended? | ❌ No | ✅ Yes |

In [ ]:
# Legacy (still works, not recommended)
np.random.seed(42)
print("Legacy rand:", np.random.rand(3))

# NEW API (recommended)
rng = np.random.default_rng(seed=42)
print("New uniform:", rng.uniform(0, 1, size=3))
print("New normal:", rng.normal(0, 1, size=3))
print("New integer:", rng.integers(0, 10, size=(2, 3)))

### 9.2 Common Distributions

| Distribution | Legacy (np.random) | New (rng) | Use Case |
| ------------ | ------------------ | ---------- | -------- |
| Uniform `[0,1)` | `rand(d0, d1, ...)` | `rng.random(size)` | Random sampling |
| Standard Normal | `randn(d0, d1, ...)` | `rng.standard_normal(size)` | Weight init |
| Normal(μ, σ) | `normal(loc, scale, size)` | `rng.normal(loc, scale, size)` | Data generation |
| Uniform(a, b) | `uniform(a, b, size)` | `rng.uniform(a, b, size)` | Param search |
| Integers [lo, hi) | `randint(lo, hi, size)` | `rng.integers(lo, hi, size)` | Indices |
| Binomial(n, p) | `binomial(n, p, size)` | `rng.binomial(n, p, size)` | ML dropout |
| Choice from array | `choice(arr, size, p=...)` | `rng.choice(arr, size, p=...)` | Sampling |
| Shuffle | `shuffle(arr)` | `rng.shuffle(arr)` | Data shuffling |
| Permutation | `permutation(n)` | `rng.permutation(n)` | Random order |

In [ ]:
rng = np.random.default_rng(2024)

# Shuffle
data = np.arange(10)
rng.shuffle(data)
print("Shuffled:", data)

# Choice with probabilities
items = np.array(['cat', 'dog', 'bird'])
probs = np.array([0.5, 0.3, 0.2])
print("Choice:", rng.choice(items, size=10, p=probs))

# Multivariate normal
mean = [0, 0]
cov = [[1, 0.8], [0.8, 1]]
samples = rng.multivariate_normal(mean, cov, size=1000)
print("\nMultivariate normal shape:", samples.shape)
print("Empirical correlation:", np.corrcoef(samples.T)[0, 1])

---
## Section 10: Advanced Topics

Deep-dive into NumPy internals and advanced usage patterns.

### 10.1 Views vs Copies — The Fundamental Distinction

Understanding when NumPy returns a **view** (shared memory) vs a **copy** (independent memory) is critical for both correctness and performance.

| Operation | Returns | Memory |
| --------- | ------- | ------ |
| Basic slicing `arr[i:j]` | **View** | Shared |
| `.reshape()` | **View** (if contiguous) | Shared |
| `.transpose()`, `.T` | **View** | Shared |
| `.ravel()` | **View** (usually) | Shared |
| Fancy indexing `arr[[0,1]]` | **Copy** | New |
| Boolean indexing `arr[mask]` | **Copy** | New |
| `.flatten()` | **Copy** | New |
| `.copy()` | **Copy** | New |
| `.astype()` | **Copy** | New |

**Check:** `np.shares_memory(a, b)` returns `True` if `b` is a view of `a`.

In [ ]:
base = np.arange(6).reshape(2, 3)
view = base[:2, :2]      # slice → view
copy = base[[0, 1]]      # fancy → copy

print("View shares memory?", np.shares_memory(base, view))
print("Copy shares memory?", np.shares_memory(base, copy))

# Modifying view vs copy
view[0, 0] = 999
print("\nBase after view modification:\n", base)

copy[0, 0] = -1
print("\nBase after copy modification (unchanged):\n", base)

### 10.2 Structured Arrays

**Definition:**  
Arrays where each element is a compound type with named fields — like a DataFrame row but in NumPy.

**Syntax:**
```python
dtype = np.dtype([('name', 'U10'), ('age', 'i4'), ('weight', 'f4')])
arr = np.array([('Alice', 30, 65.0)], dtype=dtype)
```

In [ ]:
# Define structured dtype
dtype = np.dtype([
    ('name', 'U10'),   # Unicode string, max 10 chars
    ('age', 'i4'),     # 4-byte int
    ('height', 'f4'),  # 4-byte float
    ('score', 'f8')    # 8-byte float
])

# Create structured array
data = np.array([
    ('Alice', 30, 1.70, 95.5),
    ('Bob', 25, 1.82, 87.2),
    ('Charlie', 35, 1.75, 91.3)
], dtype=dtype)

print("Structured array:")
print(data)
print("\nNames:", data['name'])
print("Mean age:", data['age'].mean())
print("Max score:", data['score'].max())

# Multi-field access
print("\nName + Score:", data[['name', 'score']])

### 10.3 Advanced NumPy Tricks

#### `np.einsum()` — Einstein Summation

**Definition:**  
A compact notation for summing over indices — can replace many combinations of transpose, multiply, and sum.

**Intuition:**  
Write the operation as a string: `'ij,jk->ik'` means matrix multiply.

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

# Matrix multiply
C1 = A @ B
C2 = np.einsum('ij,jk->ik', A, B)
print("einsum matmul:\n", C2)
print("Matches?", np.allclose(C1, C2))

# Trace
print("\nTrace via einsum:", np.einsum('ii', A))  # 5

# Outer product
v = np.array([1, 2, 3])
w = np.array([4, 5])
print("\nOuter via einsum:\n", np.einsum('i,j->ij', v, w))

# Batch matrix multiplication
batch_A = np.random.randn(10, 3, 4)
batch_B = np.random.randn(10, 4, 5)
result = np.einsum('bij,bjk->bik', batch_A, batch_B)
print("Batch matmul shape:", result.shape)  # (10, 3, 5)

#### 10.4 Memory Strides & `as_strided`

**Warning:** `np.lib.stride_tricks.as_strided` is dangerous — wrong strides cause crashes or data corruption.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

# Safe sliding window (NumPy ≥ 1.20)
arr = np.arange(10)
windows = sliding_window_view(arr, window_shape=3)
print("Sliding windows:\n", windows)

# 2D sliding window
img = np.arange(16).reshape(4, 4)
patches = sliding_window_view(img, window_shape=(2, 2))
print("\n2D sliding windows shape:", patches.shape)  # (3, 3, 2, 2)

### 10.5 `np.vectorize` & Custom ufuncs

**Definition:**  
Turn a Python function into a vectorized function. Note: it's just a loop wrapper (no speedup).

In [ ]:
# Custom function: piecewise
def piecewise_activation(x):
    if x < 0:
        return 0.0
    elif x < 1:
        return x
    else:
        return 1.0

v_activation = np.vectorize(piecewise_activation)
x = np.array([-1, 0, 0.5, 1.5, 2])
print("Vectorized:", v_activation(x))

# Better approach: use np.select
conds = [x < 0, x < 1, x >= 1]
choices = [0.0, x, 1.0]
print("np.select:", np.select(conds, choices))

### 10.6 `np.lib.scimath` — Complex Math

Functions that return complex results for negative inputs (e.g., `sqrt(-1)` → `1j`).

In [ ]:
from numpy.lib.scimath import sqrt as csqrt
print("sqrt(-4):", csqrt(-4))  # 2j
print("sqrt(4):", csqrt(4))    # 2.0

print("\nlog(-1):", np.lib.scimath.log(-1))  # πj

---
## Section 11: Interview Questions

### 🟢 Beginner

| # | Question | Answer |
| - | -------- | ------ |
| 1 | What is the difference between a Python list and a NumPy array? | NumPy arrays are typed, contiguous, support vectorized operations, and are more memory efficient. |
| 2 | How do you create a 3×3 array of zeros? | `np.zeros((3, 3))` |
| 3 | What does `arr.shape` return? | A tuple of integers representing the length of each dimension. |
| 4 | How do you convert a list `[1,2,3]` to a NumPy array? | `np.array([1, 2, 3])` |
| 5 | What is the default dtype for `np.ones(5)`? | `float64` |

### 🟡 Intermediate

| # | Question | Answer |
| - | -------- | ------ |
| 1 | What is broadcasting? | NumPy's ability to perform operations on arrays of different shapes by virtually stretching dimensions of size 1. |
| 2 | Explain `reshape(-1)`. | Flattens the array into 1D. The `-1` tells NumPy to infer the dimension. |
| 3 | What is the difference between `np.dot` and `np.matmul`? | For 2D, they're similar. For >2D, `matmul` broadcasts over batch dims; `dot` sums over the last axis of the first and second-to-last of the second. |
| 4 | How do you find indices where a condition is True? | `np.where(condition)` returns a tuple of index arrays. |
| 5 | What does `keepdims=True` do in aggregation functions? | Preserves the reduced dimension as size 1, enabling correct broadcasting. |

### 🔴 Advanced

| # | Question | Answer |
| - | -------- | ------ |
| 1 | Explain the difference between C-contiguous and F-contiguous arrays. | C-contiguous: last index changes fastest (row-major). F-contiguous: first index changes fastest (column-major). Strides differ accordingly. |
| 2 | How does `np.einsum('ij,jk->ik', A, B)` work? | It contracts index `j` — sums over `j` while preserving `i` and `k`. This is matrix multiplication. |
| 3 | What is a view vs a copy? Give 3 operations that return each. | View: slice, reshape, transpose. Copy: fancy indexing, boolean indexing, `astype()`. |
| 4 | How would you implement a sliding window (convolution) using NumPy? | Use `np.lib.stride_tricks.sliding_window_view` or `as_strided`. |
| 5 | What happens when you do `np.array([1, 2, 3])[np.array([True, False, True])]`? | It returns `array([1, 3])` — boolean indexing filters elements. |

---
## Section 12: Comparison Tables

### Array Creation Methods

| Function | Purpose | Default dtype | Complexity | Returns |
| -------- | ------- | ------------- | ---------- | ------- |
| `np.array(obj)` | Convert object to array | Inferred | O(n) | New array |
| `np.zeros(shape)` | Array of zeros | float64 | O(n) | New array |
| `np.ones(shape)` | Array of ones | float64 | O(n) | New array |
| `np.empty(shape)` | Uninitialized array | float64 | O(1) | New array (garbage) |
| `np.arange([s,]st[,st])` | Sequence | Inferred | O(n) | New array |
| `np.linspace(s,st,n)` | Evenly spaced (fixed count) | float64 | O(n) | New array |
| `np.eye(N)` | Identity matrix | float64 | O(n²) | New 2D array |
| `np.full(s, v)` | Constant-filled | Inferred | O(n) | New array |
| `np.random.rand(d0,...)` | Uniform [0,1) | float64 | O(n) | New array |
| `np.random.randn(d0,...)` | Standard normal | float64 | O(n) | New array |

### Aggregation Functions

| Function | Axis support | keepdims | NaN-safe | Output type |
| -------- | ------------ | -------- | -------- | ----------- |
| `sum()` | ✅ | ✅ | `nansum` | scalar or array |
| `mean()` | ✅ | ✅ | `nanmean` | scalar or array |
| `std()` / `var()` | ✅ | ✅ | `nanstd`/`nanvar` | scalar or array |
| `min()` / `max()` | ✅ | ✅ | `nanmin`/`nanmax` | scalar or array |
| `argmin()` / `argmax()` | ✅ | ❌ | `nanargmin`/`nanargmax` | int or array |
| `cumsum()` / `cumprod()` | ✅ (axis only) | ❌ | `nancumsum`/`nancumprod` | array |

### Reshaping & Manipulation

| Function | Returns view? | Preserves data? | Changes size? |
| -------- | ------------- | --------------- | ------------- |
| `reshape` | ✅ (if possible) | ✅ | ❌ |
| `resize` (np) | ❌ (copy) | ✅ (fills repeats) | ✅ |
| `flatten` | ❌ (copy) | ✅ | ❌ (1D) |
| `ravel` | ✅ (usually) | ✅ | ❌ (1D) |
| `transpose` | ✅ | ✅ | ❌ (permutes) |
| `concatenate` | ❌ (copy) | ✅ | ✅ |
| `stack` | ❌ (copy) | ✅ | ✅ (new axis) |
| `split` | ✅ (views) | ✅ | ✅ |

---
## Section 13: Memory Tips & Shortcuts

### 🧠 Quick Reference Card

| Task | Code |
| ---- | ---- |
| Import NumPy | `import numpy as np` |
| Create array | `np.array([1,2,3])` |
| Zeros (like shape) | `np.zeros_like(arr)` |
| Sequence | `np.arange(10)` |
| Evenly spaced | `np.linspace(0,1,5)` |
| Shape | `arr.shape` |
| Reshape | `arr.reshape(-1, 4)` |
| Transpose | `arr.T` or `arr.transpose(1,0,2)` |
| Stack vertically | `np.vstack((a,b))` |
| Stack horizontally | `np.hstack((a,b))` |
| Index of max | `arr.argmax()` |
| Where condition | `np.where(arr > 0)` |
| Filter | `arr[arr > 0]` |
| Replace NaNs | `np.nan_to_num(arr)` |
| Unique values | `np.unique(arr)` |
| Clip values | `np.clip(arr, min, max)` |
| Sort | `np.sort(arr)` |
| argsort | `np.argsort(arr)` |
| Set seed (new) | `rng = np.random.default_rng(seed)` |
| Check contiguous | `arr.flags.c_contiguous` |
| Check views | `np.shares_memory(a, b)` |
| Copy array | `arr.copy()` |
| Change dtype | `arr.astype('float32')` |
| Matrix multiply | `A @ B` |
| Element-wise multiply | `A * B` |

### ⚡ Performance Tips

1. **Avoid loops:** Never iterate over NumPy arrays in Python
2. **Pre-allocate:** `np.zeros(N)` + fill, never `np.append` in a loop
3. **Use in-place ops:** `arr += 1` instead of `arr = arr + 1`
4. **Choose order wisely:** C-order (row-major) for row-wise ops, F-order for column-wise
5. **Use `out` parameter:** `np.add(a, b, out=result)` reuses memory
6. **Avoid unnecessary copies:** Prefer slices to fancy indexing when you don't need a copy
7. **Use `np.where` for conditional fills** instead of loops
8. **Vectorize with ufuncs:** Prefer `np.sin(x)` over `[math.sin(v) for v in x]`
9. **Memory-mapped files:** Use `np.memmap` for datasets larger than RAM
10. **NumPy's `__array_function__` protocol:** Many libraries (CuPy, JAX) can replace NumPy for GPU/TPU

### 🚨 Common Mistakes Checklist

- [ ] Forgetting `np` is for *homogeneous* typed data — mixed types → `dtype=object` (slow)
- [ ] Confusing `np.arange(0, 1, 0.1)` (FP error) vs `np.linspace(0, 1, 11)` (exact)
- [ ] Assuming `arr.reshape()` returns a copy — it's usually a view
- [ ] Using `=` to copy arrays — use `.copy()` instead
- [ ] Using `and`/`or` instead of `&`/`|` with boolean arrays
- [ ] Forgetting `keepdims=True` when broadcasting after aggregation
- [ ] Not specifying `dtype` for integer np.zeros — `np.zeros(3, dtype=int)`
- [ ] Confusing `np.dot` and `*` for matrix multiplication
- [ ] Modifying a slice and being surprised the original changed
- [ ] Not seeding the RNG for reproducible experiments

---
## Section 14: Practice Problems

Try these to reinforce your learning.

In [ ]:
# Problem 1: Normalize a matrix (min-max scaling)
# Given: arr = np.random.randn(10, 10)
# Task: Scale each column to [0, 1]

arr = np.random.randn(10, 10)
col_min = arr.min(axis=0, keepdims=True)
col_max = arr.max(axis=0, keepdims=True)
normalized = (arr - col_min) / (col_max - col_min)
print("Problem 1: Normalized each col ∈ [0,1]?",
      np.allclose(normalized.min(axis=0), 0) and np.allclose(normalized.max(axis=0), 1))

In [ ]:
# Problem 2: Compute pairwise Euclidean distances
# Given: X = np.random.randn(5, 3)  (5 points, 3 dimensions)
# Task: Compute 5x5 distance matrix

X = np.random.randn(5, 3)
# Using broadcasting: ||a - b||^2 = ||a||^2 + ||b||^2 - 2a·b
sum_sq = np.sum(X**2, axis=1, keepdims=True)  # (5, 1)
dist_sq = sum_sq + sum_sq.T - 2 * (X @ X.T)
dist_sq = np.clip(dist_sq, 0, None)  # clip FP negatives
dists = np.sqrt(dist_sq)
print("Problem 2: Pairwise distances shape:", dists.shape)
print("Diagonal all zeros?", np.allclose(np.diag(dists), 0))

In [ ]:
# Problem 3: Moving average (convolution)
# Given: signal = np.sin(np.linspace(0, 10, 100)) + np.random.randn(100)*0.1
# Task: Smooth with window size 5

signal = np.sin(np.linspace(0, 10, 100)) + np.random.randn(100) * 0.1
window = np.ones(5) / 5  # uniform kernel
smoothed = np.convolve(signal, window, mode='same')
print("Problem 3: Signal shape:", signal.shape, "Smoothed shape:", smoothed.shape)

In [ ]:
# Problem 4: Image patch extraction (advanced)
# Given: img = np.arange(64).reshape(8, 8)
# Task: Extract all 3x3 patches

img = np.arange(64).reshape(8, 8)
patches = sliding_window_view(img, window_shape=(3, 3))
print("Problem 4: Patches shape:", patches.shape)  # (6, 6, 3, 3)
print("First patch (top-left):\n", patches[0, 0])

---
## Section 15: Cheat Sheet — NumPy ufuncs at a Glance

### Math
`add`, `subtract`, `multiply`, `divide`, `power`, `sqrt`, `square`, `abs`, `sign`, `mod`, `remainder`

### Trigonometry
`sin`, `cos`, `tan`, `arcsin`, `arccos`, `arctan`, `arctan2`, `degrees`, `radians`, `deg2rad`, `rad2deg`

### Exponents & Logs
`exp`, `expm1`, `exp2`, `log`, `log2`, `log10`, `log1p`

### Rounding
`round`, `floor`, `ceil`, `trunc`, `rint`, `fix`

### Comparison
`greater`, `less`, `equal`, `not_equal`, `greater_equal`, `less_equal`, `logical_and`, `logical_or`, `logical_not`, `maximum`, `minimum`, `clip`

### Floating Point
`isfinite`, `isinf`, `isnan`, `isnat`, `copysign`, `nextafter`, `spacing`, `ldexp`, `frexp`

### Set Operations
`np.unique(arr)`, `np.in1d(a, b)`, `np.intersect1d(a, b)`, `np.union1d(a, b)`, `np.setdiff1d(a, b)`, `np.setxor1d(a, b)`

---

# 🎉 Congratulations!

You've completed this comprehensive NumPy tutorial. You now have a solid foundation in:

- Array creation (10+ methods)
- Indexing (basic, fancy, boolean)
- Broadcasting and vectorization
- Universal functions (trig, exp, log, rounding)
- Linear algebra (solve, eig, SVD, norms)
- Random number generation (legacy & new API)
- Views vs copies
- Memory layout and optimization
- Advanced topics (einsum, structured arrays, sliding windows)

**Next Steps:** Practice on real datasets, explore NumPy's C API for custom extensions, or move on to SciPy, scikit-learn, and deep learning frameworks.

```
Happy Coding!  ~ The NumPy Way
```